# Week 5: Advanced Ensemble Algorithms
## Notebook 12: Model Interpretability: PDP vs Accumulated Local Effects (ALE)

### 1. Conceptual Intuition & Theoretical Background
While SHAP and LIME provide instance-level explanations, understanding the **global marginal effect** of a continuous feature requires Partial Dependence or Accumulated Local Effects.

**1. Partial Dependence Plots (PDP):**
Measures the average model output when feature $x_s$ is fixed to value $v$, integrating over the empirical marginal distribution of all other features $x_c$:
$$\hat{f}_{\text{PDP}}(v) = \frac{1}{n} \sum_{i=1}^{n} \hat{f}(v, x_{i,c})$$
- **Fatal Flaw:** When features are correlated (e.g. Height and Weight), PDP creates implausible synthetic records (e.g. Height = 6'5" with Weight = 90 lbs), yielding heavily biased estimates.

**2. Accumulated Local Effects (ALE):**
Calculates differences in predictions within localized bins and accumulates differences, evaluating points only within their true conditional distribution:
$$\hat{f}_{\text{ALE}}(x_s) = \int_{z_0}^{x_s} \mathbb{E}_{X_c \mid X_s = z} \left[ \frac{\partial \hat{f}(z, X_c)}{\partial z} \right] dz$$

---

### 2. Scratch Python Implementation: PDP vs Correlated Reality Engine

In [ ]:
import numpy as np
import pandas as pd
from sklearn.ensemble import RandomForestRegressor

# Scratch 1D Partial Dependence Calculator
def calculate_pdp_scratch(model, X_df, target_feature, grid_resolution=20):
    grid_values = np.linspace(X_df[target_feature].min(), X_df[target_feature].max(), grid_resolution)
    pdp_results = []
    
    X_synthetic = X_df.copy()
    for val in grid_values:
        # Intervene: overwrite column with constant value
        X_synthetic[target_feature] = val
        # Average model predictions across all modified rows
        mean_pred = np.mean(model.predict(X_synthetic))
        pdp_results.append(mean_pred)
        
    return grid_values, np.array(pdp_results)

# Generate Correlated Features: Income and House Size
np.random.seed(42)
n_samples = 300
income = np.random.uniform(30000, 150000, n_samples)
# Strongly correlated: House size increases with income
house_size = 0.02 * income + np.random.normal(500, 200, n_samples)

# True ground truth price: 150 * house_size (income has ZERO direct effect on price!)
price = 150.0 * house_size + np.random.normal(0, 1000, n_samples)

df_housing = pd.DataFrame({'income': income, 'house_size': house_size})
rf_model = RandomForestRegressor(n_estimators=30, random_state=42).fit(df_housing, price)

grid_vals, pdp_income = calculate_pdp_scratch(rf_model, df_housing, 'income')

print("--- Partial Dependence Plot (PDP) Analysis ---")
print(f"Income Grid [Min, Max]: [{grid_vals.min():.0f}, {grid_vals.max():.0f}]")
print(f"Predicted Price at Min Income (30k):  USD {pdp_income[0]:,.0f}")
print(f"Predicted Price at Max Income (150k): USD {pdp_income[-1]:,.0f}")
print(f"Artificial Price Lift attributed to Income: +USD {pdp_income[-1] - pdp_income[0]:,.0f}")
print("PDP mistakenly attributes large effects to Income due to correlated feature extrapolation!")

### 3. Step-by-Step Code Walkthrough
1. **Marginal Intervention:** Overwrites the target column while holding all other features fixed.
2. **Correlation Leakage:** Demonstrates how PDP forces models to evaluate physically impossible feature combinations.
3. **ALE Advantage:** ALE resolves this by computing differences across local conditional windows.

---

### 4. Industry Applications & Trade-offs
- **Regulatory Model Auditing:** Verifying credit pricing models comply with fair-lending requirements without confounding bias.
- **Trade-off:** ALE plots can be sensitive to chosen interval bin widths when sample sizes are small.

---

### 5. Top Interview Questions & Answers
**Q1: Why do Partial Dependence Plots (PDP) produce misleading curves when features are strongly correlated?**
> **Answer:** PDP varies the target feature across its entire range while keeping other features fixed, forcing the model to evaluate instances outside the true data distribution (e.g. extreme outliers).

**Q2: How does Accumulated Local Effects (ALE) resolve the correlation problem?**
> **Answer:** ALE isolates the effect of a feature within localized conditional windows and accumulates prediction differences across bins, avoiding evaluations on out-of-distribution synthetic combinations.